# Práctica 2: Aprendizaje no supervisado
## Determinación de Tipos de Estrellas

### Carga de datos y configuración inicial
Fijamos la semilla aleatoria utilizando el NIA proporcionado (100522196) tal y como se solicita en las consideraciones generales para que los resultados sean reproducibles.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Fijar la semilla con el NIA proporcionado
NIA = 100522196
np.random.seed(NIA)
random.seed(NIA)

In [ ]:
# Cargar los datos (el fichero proporcionado se llama stars_data.csv)
df = pd.read_csv('stars_data.csv')
print(f"Dimensiones del dataset: {df.shape}")
df.head()

### 1. Codificación de variables categóricas
Las variables `Spectral_Class` y `Color` son ordinales ya que están relacionadas con la energía y temperatura de la estrella. Realizamos una limpieza previa y aplicamos un *Ordinal Encoding*.

In [ ]:
# Limpieza de la columna Color
df['Color'] = df['Color'].str.lower().str.replace('-', ' ')

color_map = {
    'blue white': 'blue white',
    'blue': 'blue',
    'white': 'white',
    'whitish': 'white',
    'yellowish white': 'yellow white',
    'yellow white': 'yellow white',
    'white yellow': 'yellow white',
    'yellowish': 'yellow',
    'pale yellow orange': 'yellow orange',
    'orange': 'orange',
    'orange red': 'orange red',
    'red': 'red'
}
df['Color'] = df['Color'].map(color_map).fillna(df['Color'])

# Codificación ordinal de Spectral_Class (O: más caliente, M: más fría)
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
spectral_mapping = {clase: i for i, clase in enumerate(spectral_order)}
df['Spectral_Class_Encoded'] = df['Spectral_Class'].map(spectral_mapping)

# Codificación ordinal de Color (De más energía/azul a menos energía/rojo)
color_order = [
    'blue', 'blue white', 'white', 
    'yellow white', 'yellow', 
    'yellow orange', 'orange', 
    'orange red', 'red'
]
color_mapping = {c: i for i, c in enumerate(color_order)}
df['Color_Encoded'] = df['Color'].map(color_mapping)

display(df[['Spectral_Class', 'Spectral_Class_Encoded', 'Color', 'Color_Encoded']].head())


### 2. Reducción de Dimensionalidad con PCA
Escalamos los datos numéricos y aplicamos PCA para reducirlos a 2 componentes principales.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Seleccionar atributos para clustering (excluimos las variables categóricas originales)
features = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_Encoded', 'Color_Encoded']
X = df[features]

# Escalar los datos (esencial antes de PCA y algoritmos basados en distancias)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar PCA con 2 componentes
pca = PCA(n_components=2, random_state=NIA)
X_pca = pca.fit_transform(X_scaled)

df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

print(f"Varianza explicada por las 2 primeras componentes: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Visualizar en el espacio PCA
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', data=df, color='gray')
plt.title('Estrellas en el espacio PCA (2 Componentes)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()


### 3. Clustering
Aplicamos algoritmos de clustering (K-Means, Jerárquico y DBSCAN) sobre las componentes obtenidas por PCA.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import scipy.cluster.hierarchy as shc

# Ejemplo base con K-Means (Se debe hacer ajuste de hiperparámetros como indica el PDF)
# Evaluamos K=6 ya que en la tabla astronómica se ven ~6 tipos de estrellas distintos
kmeans = KMeans(n_clusters=6, random_state=NIA, n_init='auto')
df['KMeans_Cluster'] = kmeans.fit_predict(X_pca)

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=df, palette='viridis')
plt.title('Agrupamiento con K-Means (K=6)')
plt.show()
